# EXP-2 rerun - undiacriticised Vietnamese, gold branch fixed

`strip_row()` used to strip diacritics from `context["title"]` but leave
`supporting_facts["title"]` alone. `gold_context()` matches on
`(title, sent_id)`, so every match failed and the gold branch was handed an
**empty context on 48 of 200 questions**. The published finding - compressed
context beating gold once tone marks are stripped - was therefore measured
against nothing on a quarter of the sample.

This notebook reruns the experiment with the fix. Expect `rows_without_gold`
to read 0 in all three modes.

**Runtime:** ~30 min on a T4. Set the accelerator before running.

In [ ]:
!pip install -q "transformers==4.45.2" llmlingua rank_bm25 tiktoken python-dotenv bitsandbytes accelerate 2>&1 | tail -2
import transformers
print('transformers =', transformers.__version__)
assert transformers.__version__.startswith('4.'), (
    f'pin did not take: {transformers.__version__}. Restart session, run again.')

In [ ]:
# The repo is public now, so clone it directly - no gist, no upload.
import os, sys, subprocess, shutil

BASE = ('/kaggle/working' if os.path.isdir('/kaggle/working')
        else '/content' if os.path.isdir('/content')
        else os.getcwd())
REPO = os.path.join(BASE, 'ANLP-IterCOMP')
os.chdir(BASE)
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(['git','clone','--depth','1',
                'https://github.com/nhantrnh/ANLP-IterCOMP.git', REPO], check=True)
os.chdir(REPO)
sys.path.insert(0, 'src')

print(subprocess.run(['git','log','-1','--format=%h %s'],capture_output=True,text=True).stdout)

In [ ]:
# The published repo may not carry the fix yet, so apply it here instead of
# depending on a push. strip_row() must strip supporting_facts["title"] as well
# as context["title"], or gold_context() matches nothing and the gold branch is
# handed an empty string.
import re, io

p = 'src/scripts/exp2_undiacritised.py'
s = io.open(p, encoding='utf-8').read()

if '"supporting_facts": {**sf,' in s:
    print('fix already in the checkout')
else:
    old = """    ctx = row["context"]
    return row | {
        "question": strip_diacritics(row["question"]),
        "context": {"title": [strip_diacritics(t) for t in ctx["title"]],
                    "sentences": [[strip_diacritics(s) for s in doc]
                                  for doc in ctx["sentences"]]},
    }"""
    new = """    ctx = row["context"]
    sf = row.get("supporting_facts") or {}
    return row | {
        "question": strip_diacritics(row["question"]),
        "context": {"title": [strip_diacritics(t) for t in ctx["title"]],
                    "sentences": [[strip_diacritics(s) for s in doc]
                                  for doc in ctx["sentences"]]},
        "supporting_facts": {**sf,
                             "title": [strip_diacritics(t) for t in sf.get("title", [])]},
    }"""
    assert old in s, 'strip_row does not look as expected - inspect it by hand'
    io.open(p, 'w', encoding='utf-8').write(s.replace(old, new, 1))
    print('patch applied')

assert '"supporting_facts": {**sf,' in io.open(p, encoding='utf-8').read()
print('verified: gold branch will see stripped supporting_facts')

In [ ]:
# VimQA only - this experiment needs nothing else.
import urllib.request, os
p = 'data/vimqa/validation.parquet'
os.makedirs(os.path.dirname(p), exist_ok=True)
if not os.path.exists(p):
    urllib.request.urlretrieve(
        'https://huggingface.co/datasets/nguyenlab/vimqa/resolve/main/'
        'data/validation-00000-of-00001.parquet', p)
print(f'{p}  {os.path.getsize(p)/1e6:.1f} MB')

In [ ]:
import torch
p = torch.cuda.get_device_properties(0)
maj, _ = torch.cuda.get_device_capability(0)
print(f'{p.name} | {p.total_memory/1e9:.1f} GB')
assert maj >= 7, f'sm_{maj}x cannot run this build of PyTorch - pick a T4'
READER = 'Qwen/Qwen2.5-7B-Instruct'

In [ ]:
# == EXP-2, fixed ==
import subprocess, sys

out = 'results/exp2_undiacritised_200_7b_fixed.json'
cmd = [sys.executable, '-u', 'src/scripts/exp2_undiacritised.py',
       '--limit', '200', '--reader', 'hf', '--reader-model', READER,
       '--load-4bit', '--scorer', 'dual', '--percentile', '90', '--out', out]

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='')
proc.wait()
print('exit', proc.returncode)

In [ ]:
# The check that matters: every mode must now have a gold context.
import json
d = json.load(open(out, encoding='utf-8'))

print(f"{'mode':6s} {'IterCOMP':>9s} {'Gold':>7s} {'delta':>7s}  {'no-gold':>7s}")
for m in ('keep', 'strip', 'both'):
    r = d['results'][m]
    ic, gd = r['f1_itercomp'], r['f1_gold']
    print(f"{m:6s} {ic:9.2f} {gd:7.2f} {ic-gd:+7.2f}  {r['rows_without_gold']:7d}")

bad = [m for m in ('keep','strip','both') if d['results'][m]['rows_without_gold']]
print()
print('FAIL - still empty:', bad) if bad else print('all three modes have gold context')

print()
print('old (buggy) numbers for comparison:')
print('  keep    50.08   54.94   -4.86        0')
print('  strip   29.29   25.52   +3.78       48')
print('  both    30.25   24.43   +5.82       48')

In [ ]:
# Save the result so it can be committed back to the repo.
import shutil, os
dst = '/kaggle/working/exp2_undiacritised_200_7b_fixed.json'
if os.path.abspath(out) != dst:
    shutil.copy(out, dst)
print('download this file:', dst)